# Niche discovery from spatial molecular context

Read the preceding step's real AnnData, aggregate molecular features within tissue cores, apply or fit the CellCharter mixture, and save the resulting labels.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import squidpy as sq
import torch
from cellcharter.gr import aggregate_neighbors
from cellcharter.tl import ClusterAutoK
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/01_domain_discovery").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "domain_reference_v2")
OUT = ROOT / "outputs/domain_discovery" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
SEED = 42
INPUT_H5AD = OUT / "01_embedding.h5ad"
OUTPUT_H5AD = OUT / "02_niches.h5ad"
MODEL_SOURCE = "reference"
REFERENCE_MODEL = ROOT / "models/autok_niche"
NEW_MODEL = OUT / "niche_exploratory_model"
EXPECTED_MODEL_SHA256 = (
    "839fc5870fd4efea90f5826f65904d05051a7c591fc33a10890c0763454f9aa6"
)
CURATED_NAMES = {
    "0": "Inflamed gastric mucosal",
    "1": "TLS-like structure",
    "2": "Fibrotic stroma",
    "3": "Smooth muscle",
    "4": "Low quality",
    "5": "Inflammation-infiltrated stroma",
    "6": "Low quality",
    "7": "Gland stroma",
    "8": "Neck cell enriched",
    "9": "Gastric-type Tumor",
    "10": "Plasma enriched stroma",
    "11": "Chief cell enriched",
    "12": "Normal vasculature",
    "13": "Invasive Tumor region",
    "14": "TAM enriched",
    "15": "TAN & Mono enriched",
    "16": "CAF enriched",
    "17": "Tumor vasculature",
    "18": "Invasive Tumor nest",
    "19": "Homeostatic stroma",
    "20": "Tumor nest",
    "21": "Epithelium",
}
GROUP_KEY = "core_id"
N_LAYERS = 3
CANDIDATE_K = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
SELECTED_K = 22
EXCLUDED_BATCHES = ("proseg_10_annotated",)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc.spatial.curation import artifact_tree_sha256, assign_curated_names

Read the previous output and check the metadata and physical coordinates used below.

In [ ]:
if not INPUT_H5AD.is_file():
    raise FileNotFoundError(f"Run the preceding step first: {INPUT_H5AD}")
data = ad.read_h5ad(INPUT_H5AD)
for key in [GROUP_KEY, "batch", "cell_type"]:
    if key not in data.obs or data.obs[key].isna().any():
        raise ValueError(f"Missing or incomplete cell metadata: {key}")
if not data.obs_names.is_unique or not data.var_names.is_unique:
    raise ValueError("Cell and gene identifiers must be unique.")
xy = np.asarray(data.obsm["spatial"])
if xy.shape != (data.n_obs, 2) or not np.isfinite(xy).all():
    raise ValueError("Expected finite physical XY coordinates for every cell.")
print(f"Loaded {data.n_obs:,} cells from {INPUT_H5AD}")

## 2. Construct graphs and aggregate local features

Build independent within-core Delaunay graphs. Concatenate the focal-cell features and mean features at graph distances one to 3; no edge connects separate cores.

In [ ]:
features = np.asarray(data.obsm["X_scVI"])
if (
    features.ndim != 2
    or features.shape[0] != data.n_obs
    or (not np.isfinite(features).all())
):
    raise ValueError("Invalid input features for spatial aggregation.")
data.obs[GROUP_KEY] = (
    data.obs[GROUP_KEY].astype("category").cat.remove_unused_categories()
)
sq.gr.spatial_neighbors(
    data,
    library_key=GROUP_KEY,
    coord_type="generic",
    delaunay=True,
    spatial_key="spatial",
)
aggregate_neighbors(
    data,
    n_layers=N_LAYERS,
    use_rep="X_scVI",
    out_key="X_cellcharter",
    sample_key=GROUP_KEY,
)
print("Aggregated feature shape:", data.obsm["X_cellcharter"].shape)

## 3. Apply or fit the mixture model

Apply the supplied fitted mixture by default. An explicitly selected refit searches the declared component counts and saves a new model; its numeric components do not inherit historical biological names.

Verify the saved mixture model before applying its curated biological names. Component numbers from a different fit are not interchangeable.

In [ ]:
if MODEL_SOURCE == "reference":
    if data.uns.get("spagc_embedding", {}).get("source") == "newly_fitted_scvi":
        raise ValueError(
            "New scVI features require a compatible newly fitted and curated mixture."
        )
    fingerprint = artifact_tree_sha256(REFERENCE_MODEL)
    model = ClusterAutoK.load(str(REFERENCE_MODEL))
elif MODEL_SOURCE == "fit":
    if NEW_MODEL.exists():
        raise FileExistsError(f"Model output already exists: {NEW_MODEL}")
    keep = ~data.obs["batch"].astype(str).isin(EXCLUDED_BATCHES)
    if not keep.any():
        raise ValueError("No cells remain after fitting exclusions.")
    model = ClusterAutoK(
        n_clusters=CANDIDATE_K,
        max_runs=10,
        convergence_tol=0.001,
        model_params={"random_state": SEED},
    )
    model.fit(data[keep].copy(), use_rep="X_cellcharter")
    NEW_MODEL.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(NEW_MODEL))
    fingerprint = artifact_tree_sha256(NEW_MODEL)
else:
    raise ValueError("MODEL_SOURCE must be reference or fit.")
model.best_models[SELECTED_K].trainer_params.update(
    accelerator="cpu" if DEVICE == "cpu" else "gpu",
    devices=1 if DEVICE == "cpu" else [int(DEVICE.split(":")[-1])],
)
codes = np.asarray(model.predict(data, use_rep="X_cellcharter", k=SELECTED_K)).astype(
    str
)
data.obs["niche_cluster"] = pd.Categorical(codes)

## 4. Annotate components and inspect their composition

Use the curated dictionary only with its verified reference model. Refitted components remain unannotated until biological curation. Inspect composition before proceeding.

Map mixture components to curated names only when model identity and the component dictionary agree.

In [ ]:
if MODEL_SOURCE == "reference":
    data.obs["niche"] = assign_curated_names(
        codes, CURATED_NAMES, fingerprint, EXPECTED_MODEL_SHA256
    )
else:
    if "niche" in data.obs:
        del data.obs["niche"]
    OUTPUT_H5AD = OUT / "niche_unannotated.h5ad"
composition = pd.crosstab(
    data.obs["niche_cluster"], data.obs["cell_type"], normalize="index"
)
if not np.isfinite(data.obsm["X_cellcharter"]).all():
    raise ValueError("Aggregated features contain non-finite values.")
display(composition)

## 5. Save the output data

Save the cell-level AnnData and the component-composition table without overwriting earlier outputs.

In [ ]:
COMPOSITION_CSV = OUT / "niche_composition.csv"
for output in [OUTPUT_H5AD, COMPOSITION_CSV]:
    if output.exists():
        raise FileExistsError(f"Output already exists: {output}")
OUT.mkdir(parents=True, exist_ok=True)
data.write_h5ad(OUTPUT_H5AD)
composition.to_csv(COMPOSITION_CSV)
print(OUTPUT_H5AD)